<div dir="rtl" style="text-align: right;">

# بنام خدا

## آزمون پایان دورهٔ تابستانهٔ دومین دورهٔ المپیاد هوش مصنوعی سال ۱۴۰۵

**نام:** ........................................

**نام خانوادگی:** ........................................

</div>

<div dir="rtl" style="text-align: right;">

## پایش چندوظیفه‌ای راکتور شیمیایی

تصور کنید مسئول طراحی یک سامانهٔ پایش (Monitoring System) برای یک واحد تولید مواد شیمیایی هستید. راکتور (Reactor) مواد اولیه را به محصول تبدیل می‌کند و تجهیزات جانبی، از جمله کندانسور (Condenser) و مدارهای آب خنک‌کننده (Cooling Water)، دما و شرایط عملیاتی را کنترل می‌کنند. اپراتور در هر لحظه به اندازه‌گیری‌های متعدد فرایند (Process Measurements) دسترسی دارد، اما بررسی تک‌تک آن‌ها برای تشخیص سریع علت یک تغییر غیرعادی ساده نیست.

دو وضعیت را در نظر بگیرید: در یکی، دمای آب خنک‌کنندهٔ ورودی راکتور به‌صورت ناگهانی تغییر کرده و در دیگری شیر (Valve) مدار خنک‌کننده گیر کرده است. هر دو ممکن است روی اندازه‌گیری‌های فرایند اثر بگذارند، اما علت و اقدام اصلاحی آن‌ها یکسان نیست. از طرف دیگر، تغییرات فرایند می‌تواند با تغییر غلظت محصول نیز همراه باشد؛ بنابراین تنها تشخیص «عادی یا غیرعادی» برای پایش کامل واحد کافی نیست.

برای این سؤال از داده‌های شبیه‌سازی (Simulation) **Tennessee Eastman Process (TEP)** استفاده می‌کنیم. در اجرای شبیه‌سازی، شرایط عادی و چند نوع اختلال (Fault) مرتبط با دمای آب خنک‌کننده و عملکرد شیر بررسی شده و متغیرهای فرایند ثبت شده‌اند. از شما خواسته می‌شود **یک مدل Multi-Task MLP با بازنمایی مشترک (Shared Representation)** طراحی کنید که از روی اطلاعات در دسترس، هم‌زمان این سه خروجی را تولید کند:

- **رگرسیون (Regression):** تخمین مقدار عددی `product_g`، که اندازه‌گیری مربوط به غلظت جزء G در محصول است.
- **دسته‌بندی دودویی (Binary Classification):** پیش‌بینی اینکه فرایند در وضعیت عادی است یا خطا دارد (`has_fault`).
- **دسته‌بندی چندکلاسه (Multi-Class Classification):** تشخیص سناریوی اختلال (`fault_type`) از میان پنج نوع تعریف‌شده.

هدف این است که یک مدل واحد، به کمک اطلاعات مشترک فرایند، هر سه وظیفه (Task) را انجام دهد. انتخاب شیوهٔ آماده‌سازی داده (Data Preprocessing)، معماری (Architecture)، تابع هدف (Objective) و روند آموزش (Training) با شماست.

تمام کتابخانه‌ها و ابزارهای موردنیاز برای یک راه‌حل استاندارد در ابتدای نوت‌بوک import شده‌اند و انتظار نمی‌رود برای حل سؤال به کتابخانه‌ای خارج از این مجموعه نیاز داشته باشید.

</div>

<div dir="rtl" style="text-align: right;">

## ۱. شناسنامهٔ داده‌ها (Data Card)

این داده‌ها از شبیه‌سازی یک واحد شیمیایی بر پایهٔ **Tennessee Eastman Process (TEP)** به‌دست آمده‌اند. در شبیه‌سازی، برخی اجراها در وضعیت عادی انجام شده‌اند و در برخی دیگر، یکی از پنج اختلال (Fault) مشخص در مدارهای خنک‌کننده یا شیر کنترلی رخ داده است. در طول اجرا، حسگرها (Sensors) و متغیرهای کنترلی (Manipulated Variables) مقادیر فرایند را در زمان‌های متوالی ثبت کرده‌اند. سپس بخشی از این اندازه‌گیری‌ها به شکل ردیف‌های یک مجموعه‌داده (Dataset) گردآوری شده‌اند. هر ردیف یک مشاهده (Observation) از وضعیت واحد است؛ مدل باید با تکیه بر اندازه‌گیری‌های موجود، بدون دریافت نام اختلال به‌عنوان ورودی، سه خروجی خواسته‌شده را تخمین بزند.

برای اپراتور، تفکیک اختلال دمای آب خنک‌کنندهٔ راکتور از اختلال کندانسور یا گیرکردن شیر اهمیت دارد؛ زیرا منشأ مسئله و اقدام اصلاحی یکسان نیست. در کنار تشخیص وضعیت و منشأ خطا، مدل باید غلظت جزء `G` در محصول را نیز پیش‌بینی کند.

پنج نام مجاز برای `fault_type` و معنای آن‌ها عبارت‌اند از:

| برچسب متنی (Label) | سناریوی شبیه‌سازی‌شده |
|---|---|
| `reactor_temp_step` | تغییر ناگهانی (Step) دمای آب خنک‌کنندهٔ ورودی راکتور |
| `condenser_temp_step` | تغییر ناگهانی (Step) دمای آب خنک‌کنندهٔ ورودی کندانسور |
| `reactor_temp_random` | نوسان تصادفی (Random Variation) دمای آب خنک‌کنندهٔ ورودی راکتور |
| `condenser_temp_random` | نوسان تصادفی (Random Variation) دمای آب خنک‌کنندهٔ ورودی کندانسور |
| `reactor_valve_stick` | گیرکردن شیر (Valve Sticking) آب خنک‌کنندهٔ راکتور |

فایل `train.csv` شامل اندازه‌گیری‌ها و برچسب‌های آموزش است:

| ستون | نقش و معنی |
|---|---|
| `id` | شناسهٔ یکتای مشاهده در فایل |
| `xmeas_*` | اندازه‌گیری‌های عددی حسگرهای فرایند |
| `xmv_*` | مقادیر عددی متغیرهای کنترلی (Manipulated Variables) فرایند |
| `product_g` | مقدار پیوستهٔ غلظت جزء G در محصول؛ هدف رگرسیون (Regression Target) |
| `has_fault` | برچسب متنی وضعیت کلی: `normal` یا `fault` |
| `fault_type` | یکی از پنج نام متنی بالا؛ برای مشاهده‌های عادی، این فیلد خالی است |

در این نسخه، **۴۳ ویژگی عددی (Numerical Features)** از خانواده‌های `xmeas_*` و `xmv_*` در اختیار شماست.

**محدودیت‌ها (Constraints):** یک شبکه با بخش مشترک (Shared Trunk) و سه خروجی (Head) متناسب با سه وظیفهٔ (Task) سؤال بسازید.

</div>

<div dir="rtl" style="text-align: right;">

## ۲. شیوهٔ ارزیابی و نمره‌دهی (Evaluation & Scoring)

داور (Grader) سه خروجی را روی دادهٔ آزمون (Test Data) ارزیابی می‌کند. برای پیش‌بینی `product_g`، معیار (Metric) **RMSE** در مقیاس اصلی است و مقدار کمتر بهتر است. برای `has_fault` و `fault_type`، معیار **Macro-F1** است و مقدار بیشتر بهتر است.

| وظیفه (Task) | معیار (Metric) | شرط امتیاز صفر | شرط امتیاز کامل | سهم از نمره |
|---|---|---|---|---|
| پیش‌بینی غلظت جزء G در محصول | RMSE | $R\ge R_{\mathrm{base}}$ | $R\le R_{\mathrm{target}}$ | یک‌سوم |
| تشخیص عادی یا خطا (Fault Detection) | Macro-F1 روی تمام ردیف‌ها | $F_{\mathrm{bin}}\le F_{\mathrm{bin,base}}$ | $F_{\mathrm{bin}}\ge F_{\mathrm{bin,target}}$ | یک‌سوم |
| تشخیص نوع خطا (Fault Classification) | Macro-F1 روی ردیف‌های دارای خطای واقعی | $F_{\mathrm{type}}\le F_{\mathrm{type,base}}$ | $F_{\mathrm{type}}\ge F_{\mathrm{type,target}}$ | یک‌سوم |

برای هر وظیفه (Task) یک امتیاز نرمال‌شده (Normalized Score) در بازهٔ صفر تا یک ساخته می‌شود. این سه مقدار را به‌ترتیب `r`، `b` و `f` می‌نامیم. مقادیر بین مرجع پایه (Baseline) و مرجع هدف (Target) به‌صورت خطی امتیاز می‌گیرند و مقادیر بیرون از بازه با تابع clip در نزدیک‌ترین کران قرار می‌گیرند. مقدار عددی آستانه‌ها (Thresholds) در داور نگهداری می‌شود.

</div>

$$
\operatorname{clip}_{[0,1]}(x)=\min\!\left(1,\max(0,x)\right)
\qquad\text{و}\qquad
\begin{aligned}
r&=\operatorname{clip}_{[0,1]}\!\left(\frac{R_{\mathrm{base}}-R}{R_{\mathrm{base}}-R_{\mathrm{target}}}\right),\\[7pt]
b&=\operatorname{clip}_{[0,1]}\!\left(\frac{F_{\mathrm{bin}}-F_{\mathrm{bin,base}}}{F_{\mathrm{bin,target}}-F_{\mathrm{bin,base}}}\right),\\[7pt]
f&=\operatorname{clip}_{[0,1]}\!\left(\frac{F_{\mathrm{type}}-F_{\mathrm{type,base}}}{F_{\mathrm{type,target}}-F_{\mathrm{type,base}}}\right).
\end{aligned}
$$

$$
\boxed{\mathrm{Score}=\frac{100}{3}\,(r+b+f)}
\qquad 0\le \mathrm{Score}\le 100
$$

<div dir="rtl" style="text-align: right;">

در فرمول بالا، `R` مقدار RMSE، `F_bin` مقدار Macro-F1 تشخیص عادی/خطا و `F_type` مقدار Macro-F1 نوع خطاست. هر وظیفه (Task) حداکثر `100/3` نمره دارد و نمرهٔ کل از ۱۰۰ است.

`fault_type` فقط روی ردیف‌هایی سنجیده می‌شود که **برچسب واقعی (Ground Truth)** آن‌ها یکی از پنج نوع خطاست؛ این معیار مستقل از پیش‌بینی `has_fault` است. برچسب‌های متنی باید دقیقاً مطابق جدول سؤال باشند.

</div>

<div dir="rtl" style="text-align: right;">

## ۳. شناخت داده و بررسی اولیه (EDA)

ابتدا فقط دادهٔ آموزشی (Training Data) را بخوانید. در تحلیل اکتشافی داده (Exploratory Data Analysis)، ساختار ستون‌ها، دامنه و توزیع متغیرهای هدف (Targets)، نسبت کلاس‌ها (Class Balance) و هر نکتهٔ دیگری را که ممکن است روی مدل یا ارزیابی اثر بگذارد بررسی کنید.

</div>

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, mean_squared_error, r2_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

TRAIN_PATH = "train.csv"
train_df = pd.read_csv(TRAIN_PATH)

print("Train shape:", train_df.shape)
display(train_df.head())
display(train_df.dtypes.to_frame("dtype").T)
print("Missing cells by column:")
display(train_df.isna().sum().sort_values(ascending=False).head(12))

In [ ]:
# Examine the feature and target distributions, label values,
# missing values, and any other data properties that may affect the model.

# YOUR CODE HERE

<div dir="rtl" style="text-align: right;">

## ۴. انتخاب دادهٔ اعتبارسنجی (Validation)

بخشی از `train_df` را با `SEED` برای اعتبارسنجی (Validation) کنار بگذارید و در تمام مراحل از همین تقسیم (Split) استفاده کنید.

</div>

In [ ]:
# Create reproducible training and validation DataFrames.
# Use random_state=SEED (or an equivalent seeded procedure).
train_part = ...
val_part = ...

# Verify that the two subsets have the expected shapes and labels.

<div dir="rtl" style="text-align: right;">

## ۵. آماده‌سازی ورودی‌ها و خروجی‌ها (Preprocessing)

برچسب‌های `has_fault` و `fault_type` متنی هستند. در `fault_type`، ردیف‌های `normal` برچسب معتبر ندارند؛ تابع خطا (Loss) هر وظیفه (Task) را تنها روی نمونه‌هایی محاسبه کنید که target معتبر آن وظیفه را دارند. نحوهٔ کدگذاری (Encoding)، مقیاس‌بندی (Scaling) و پوشاندن (Masking) با شماست.

</div>

In [ ]:
# Choose the model input columns and prepare model inputs.
feature_cols = ...
X_train = ...
X_val = ...

# Prepare the targets for the three tasks.
y_product_train = ...
y_product_val = ...
y_binary_train = ...
y_binary_val = ...
y_fault_train = ...
y_fault_val = ...

# Store any preprocessing objects needed for later inference.
# Build suitable PyTorch datasets and dataloaders.
BATCH_SIZE = ...
train_loader = ...
val_loader = ...

<div dir="rtl" style="text-align: right;">

## ۶. معماری مدل چندوظیفه‌ای (Multi-Task Architecture)

با استفاده از یک بازنمایی مشترک (Shared Representation)، سه خروجی (Head) متناسب با سه وظیفه (Task) بسازید. ابعاد، تعداد لایه‌ها (Layers) و ساختار هر head با شماست. معماری و تابع `forward` را کامل کنید.

</div>

In [ ]:
class MultiTaskMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        # Define the complete shared network and the three task heads.
        # YOUR CODE HERE

    def forward(self, x):
        # Return the three task outputs in this order:
        # product_output, binary_output, fault_output.
        raise NotImplementedError("Implement MultiTaskMLP.forward")

<div dir="rtl" style="text-align: right;">

## ۷. تنظیمات مدل و تابع هدف (Objective)

مدل را ایجاد کنید و بهینه‌ساز (Optimizer) و تابع هدف (Objective) لازم برای آموزش هم‌زمان سه وظیفه (Task) را مشخص کنید. نوع خروجی headها و توابع خطا (Loss Functions) را با توجه به targetها انتخاب کنید.

</div>

In [ ]:
device = torch.device("cpu")

# Set the model input dimension and instantiate the network.
model = ...

# Define the optimizer and the loss functions for all three tasks.
optimizer = ...
product_loss_fn = ...
binary_loss_fn = ...
fault_loss_fn = ...

# Configure any additional training parameters if needed.

<div dir="rtl" style="text-align: right;">

## ۸. یک Epoch آموزش (Training)

دادهٔ هر batch را باز کنید و loss سه وظیفه (Task) را در یک loss کل ترکیب کنید؛ loss مربوط به `fault_type` فقط روی ردیف‌های دارای برچسب خطا حساب شود.

</div>

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    n_batches = 0

    for batch in loader:
        # Unpack one batch from your dataset.
        x = ...
        y_product = ...
        y_binary = ...
        y_fault = ...

        x = x.to(device)
        y_product = y_product.to(device)
        y_binary = y_binary.to(device)
        y_fault = y_fault.to(device)

        optimizer.zero_grad()
        product_output, binary_output, fault_output = model(x)

        # Compute the task losses and combine them into one scalar.
        total_loss = ...

        total_loss.backward()
        optimizer.step()

        running_loss += float(total_loss.item())
        n_batches += 1

    return {"loss": running_loss / max(n_batches, 1)}

<div dir="rtl" style="text-align: right;">

## ۹. ارزیابی روی Validation (Evaluation)

خروجی خام (Raw Output) شبکه را به پیش‌بینی تبدیل کنید و عملکرد هر سه وظیفه (Task) را جداگانه بسنجید. معیار `product_g` مقدار **RMSE در مقیاس اصلی** است و دو وظیفهٔ (Task) دسته‌بندی (Classification) با **Macro-F1** سنجیده می‌شوند. برای `fault_type`، Macro-F1 فقط روی مشاهداتی محاسبه می‌شود که نوع خطای واقعیِ آن‌ها یکی از پنج برچسب تعریف‌شده است.

</div>

In [ ]:
@torch.no_grad()
def evaluate_model(model, loader, device):
    model.eval()
    product_true_all, product_pred_all = [], []
    binary_true_all, binary_pred_all = [], []
    fault_true_all, fault_pred_all = [], []

    for batch in loader:
        # Unpack the current batch as in the training function.
        x = ...
        y_product = ...
        y_binary = ...
        y_fault = ...

        product_output, binary_output, fault_output = model(x.to(device))

        # Convert raw outputs into evaluation-ready predictions.
        # The regression values must be in the original product_g units.
        product_pred = ...
        binary_pred = ...
        fault_pred = ...

        # Ensure target values are in the appropriate representation.
        product_true = ...
        binary_true = ...
        fault_true = ...

        product_true_all.append(np.asarray(product_true).reshape(-1))
        product_pred_all.append(np.asarray(product_pred).reshape(-1))
        binary_true_all.append(np.asarray(binary_true).reshape(-1))
        binary_pred_all.append(np.asarray(binary_pred).reshape(-1))
        fault_true_all.append(np.asarray(fault_true).reshape(-1))
        fault_pred_all.append(np.asarray(fault_pred).reshape(-1))

    arrays = {
        "product_true": np.concatenate(product_true_all),
        "product_pred": np.concatenate(product_pred_all),
        "binary_true": np.concatenate(binary_true_all),
        "binary_pred": np.concatenate(binary_pred_all),
        "fault_true": np.concatenate(fault_true_all),
        "fault_pred": np.concatenate(fault_pred_all),
    }

    # Compute RMSE and both Macro-F1 values from the arrays.
    metrics = ...
    return {"metrics": metrics, **arrays}

<div dir="rtl" style="text-align: right;">

## ۱۰. آموزش و انتخاب مدل (Model Selection)

مدل را آموزش دهید و عملکرد Validation را در پایان هر epoch ببینید. نحوهٔ استفاده از دادهٔ آموزشی برای ساخت پیش‌بینی نهایی با شماست.

</div>

In [ ]:
EPOCHS = ...
history = []

for epoch in range(EPOCHS):
    train_result = train_one_epoch(
        model=model, loader=train_loader, optimizer=optimizer, device=device
    )
    validation_result = evaluate_model(
        model=model, loader=val_loader, device=device
    )
    history.append({
        "epoch": epoch + 1,
        "train": train_result,
        "validation": validation_result["metrics"],
    })
    print(
        f"Epoch {epoch + 1:02d} | "
        f"train_loss={train_result['loss']:.4f} | "
        f"validation={validation_result['metrics']}"
    )

<div dir="rtl" style="text-align: right;">

## ۱۱. پیش‌بینی روی دادهٔ آزمون (Test)

عملکرد مدل روی دادهٔ آزمون (Test) سنجیده می‌شود؛ با همان پیش‌پردازش (Preprocessing) برای تمام ردیف‌های `test.csv` پیش‌بینی بسازید؛ خروجی دو وظیفهٔ (Task) دسته‌بندی باید برچسب متنی باشد.

</div>

In [ ]:
TEST_PATH = "test.csv"
test_df = pd.read_csv(TEST_PATH)
print("Test shape:", test_df.shape)

# Apply the selected preprocessing to test_df without data leakage.
# Run your final model and obtain predictions for all three tasks.

# Store one-dimensional arrays with exactly one prediction per test row.
# product_predictions must be returned to the ORIGINAL product_g scale.
product_predictions = ...
binary_predictions_text = ...
fault_predictions_text = ...

<div dir="rtl" style="text-align: right;">

## ۱۲. ساخت فایل نهایی تحویل (Submission)

پیش از اجرای سلول آخر، همین نوت‌بوک تکمیل‌شده را با نام `chemical-reactor-multitask-monitoring.ipynb` در پوشهٔ جاری ذخیره کنید.

- سلول آخر شکل و تعداد پیش‌بینی‌ها، مقادیر نامعتبر، شناسه‌ها و برچسب‌های مجاز را بررسی می‌کند.
- فایل `submission.csv` را می‌سازد.
- فایل `submission.zip` را می‌سازد که دقیقاً شامل `submission.csv` و همین نوت‌بوک تکمیل‌شده است.

فایل `submission.zip` را بارگذاری کنید، نه فایل CSV را به‌تنهایی.

</div>

In [ ]:
from pathlib import Path
from zipfile import ZIP_DEFLATED, ZipFile

NOTEBOOK_FILENAME = "chemical-reactor-multitask-monitoring.ipynb"
CSV_FILENAME = "submission.csv"
ZIP_FILENAME = "submission.zip"

product_predictions = np.asarray(product_predictions)
binary_predictions_text = np.asarray(binary_predictions_text)
fault_predictions_text = np.asarray(fault_predictions_text)

for name, values in {
    "product_predictions": product_predictions,
    "binary_predictions_text": binary_predictions_text,
    "fault_predictions_text": fault_predictions_text,
}.items():
    if values.ndim != 1:
        raise ValueError(f"{name} must be one-dimensional.")
    if len(values) != len(test_df):
        raise ValueError(f"{name} count does not match test row count.")
    if pd.isna(values).any():
        raise ValueError(f"{name} contains missing values.")

if not np.isfinite(product_predictions.astype(float)).all():
    raise ValueError("product_g predictions must be finite numbers.")

allowed_binary = {"normal", "fault"}
allowed_faults = {
    "reactor_temp_step",
    "condenser_temp_step",
    "reactor_temp_random",
    "condenser_temp_random",
    "reactor_valve_stick",
}
if not set(binary_predictions_text.tolist()) <= allowed_binary:
    raise ValueError("has_fault contains an invalid label.")
if not set(fault_predictions_text.tolist()) <= allowed_faults:
    raise ValueError("fault_type contains an invalid label.")

if test_df["id"].isna().any() or test_df["id"].duplicated().any():
    raise ValueError("Test identifiers must be present and unique.")

submission = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "product_g": product_predictions.astype(float),
    "has_fault": binary_predictions_text,
    "fault_type": fault_predictions_text,
})
expected_columns = ["id", "product_g", "has_fault", "fault_type"]
if submission.columns.tolist() != expected_columns:
    raise ValueError("Submission columns are invalid.")
if not submission["id"].equals(test_df["id"]):
    raise ValueError("Submission identifiers must match test.csv in order.")

notebook_path = Path(NOTEBOOK_FILENAME)
if not notebook_path.is_file():
    raise FileNotFoundError(
        f"Save this notebook as {NOTEBOOK_FILENAME} in the current folder, then run this cell again."
    )

submission.to_csv(CSV_FILENAME, index=False)
with ZipFile(ZIP_FILENAME, "w", compression=ZIP_DEFLATED) as archive:
    archive.write(CSV_FILENAME, arcname=CSV_FILENAME)
    archive.write(notebook_path, arcname=NOTEBOOK_FILENAME)

print(f"{ZIP_FILENAME} created successfully.")
display(submission.head())